In [ ]:
from flask import Flask, request, Response, render_template, jsonify
import cv2
import numpy as np
import threading
import time
from datetime import datetime

app = Flask(__name__)

latest_frame = None
data_lock = threading.Lock()

latest_status = {
    "last_frame_ts": None,
    "last_command_ts": None,
    "classes": {
        "person": 0,
        "falldown": 0,
        "attack": 0,
        "smoking": 0
    }
}


def format_ts(ts):
    if ts is None:
        return "-"
    return datetime.fromtimestamp(ts).strftime("%Y-%m-%d %H:%M:%S")


def safe_int(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return 0


def normalize_situation(data):
    situation = data.get("situation", {})

    if not isinstance(situation, dict):
        situation = {}

    return {
        "person": safe_int(situation.get("person", 0)),
        "falldown": safe_int(situation.get("falldown", 0)),
        "attack": safe_int(situation.get("attack", 0)),
        "smoking": safe_int(situation.get("smoking", 0)),
    }



# 비디오 업로드
@app.route('/upload', methods=['POST'])
def upload():
    global latest_frame

    nparr = np.frombuffer(request.data, np.uint8)
    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    if frame is None:
        return "Invalid Frame", 400

    ret, jpeg = cv2.imencode('.jpg', frame)
    if not ret:
        return "JPEG Encode Failed", 500

    with data_lock:
        latest_frame = jpeg.tobytes()
        latest_status["last_frame_ts"] = time.time()

    return "OK", 200


# 라즈베리파이가 보낸 JSON 데이터
@app.route('/command', methods=['POST'])
def receive_command():
    data = request.get_json(silent=True)

    if data is None:
        return jsonify({"status": "error", "message": "Invalid JSON"}), 400

    counts = normalize_situation(data)

    with data_lock:
        latest_status["classes"] = counts
        latest_status["last_command_ts"] = time.time()

    print(f"[서버 수신] 명령어 도착: {counts}")
    print(f"person 개수 {counts['person']}")
    print(f"falldown 개수 {counts['falldown']}")
    print(f"attack 개수 {counts['attack']}")
    print(f"smoking 개수 {counts['smoking']}")

    return jsonify({
        "status": "success",
        "received_action": counts
    }), 200



# 웹페이지 상태 조회 API
@app.route('/api/status')
def api_status():
    with data_lock:
        now = time.time()
        last_frame_ts = latest_status["last_frame_ts"]
        camera_online = last_frame_ts is not None and (now - last_frame_ts < 3.0)

        classes = latest_status["classes"].copy()
        last_frame_time = format_ts(latest_status["last_frame_ts"])
        last_command_time = format_ts(latest_status["last_command_ts"])

    active_alerts = []
    if classes["falldown"] > 0:
        active_alerts.append(f"쓰러짐 {classes['falldown']}건")
    if classes["attack"] > 0:
        active_alerts.append(f"공격 {classes['attack']}건")
    if classes["smoking"] > 0:
        active_alerts.append(f"흡연 {classes['smoking']}건")

    return jsonify({
        "camera_online": camera_online,
        "last_frame_time": last_frame_time,
        "last_command_time": last_command_time,
        "classes": classes,
        "active_alerts": active_alerts
    })



# 실시간 영상 스트리밍
def generate():
    global latest_frame

    while True:
        with data_lock:
            frame = latest_frame

        if frame is not None:
            yield (
                b'--frame\r\n'
                b'Content-Type: image/jpeg\r\n\r\n' + frame + b'\r\n'
            )

        time.sleep(0.03)


@app.route('/video_feed')
def video_feed():
    return Response(generate(), mimetype='multipart/x-mixed-replace; boundary=frame')



# 메인 페이지
@app.route('/')
def index():
    return render_template('index.html')


if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False, threaded=True)